___
# <center>Exercícios: medidas de erro e regressão logística</center>
___

## Aula 22

**Objetivo:** ao final destes exercícios, você deve ser capaz de:

 * calcular acurácia, sensibilidade, especificidade e precisão a partir de uma
   matriz de confusão, e dizer o que cada uma significa no caso;
 * escolher a medida que mais importa para um problema, e o ponto de corte;
 * interpretar a saída de uma regressão logística, inclusive com variável
   categórica, e calcular uma probabilidade prevista a partir dela;
 * explicar por que um coeficiente muda quando entra outra variável;
 * desconfiar de um modelo que acerta muito nos dados em que foi ajustado.

**Vale até 4 pontos extras no Projeto 06**, 1 por exercício. É individual:
pode conversar com colegas, mas escreva com as suas palavras.

**Você não precisa escrever código de modelo.** As células de código já estão
prontas: é só rodar. Onde houver conta, use o Python como calculadora (por
exemplo, `152 / 201`).

**Nome:**

**E-mail Insper:**

**Entrega:** ao terminar, baixe este notebook (*Arquivo > Fazer download >
Baixar o .ipynb*) e envie no BlackBoard, no item **Exercícios da aula 22**.

___
<div id="indice"></div>

## Índice

- [As bases](#bases)
- [Exercício 1: reclamações trabalhistas, quantas terminam em acordo?](#ex1)
- [Exercício 2: pedidos de benefício, onde está a fraude?](#ex2)
- [Exercício 3: ações contra companhias aéreas, o que explica a procedência?](#ex3)
- [Exercício 4: juizado especial, e se o modelo souber quem é o juiz?](#ex4)

___
<div id="bases"></div>

## As bases

**Todas as bases são simuladas.** Os números ilustram os métodos e não dizem
nada sobre processos reais. Rode a célula abaixo antes de começar.

In [ ]:
# @title Rode esta célula para criar as bases (não precisa ler o código)
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

rng = np.random.default_rng(2026)
sig = lambda z: 1 / (1 + np.exp(-z))

# exercício 1: reclamações trabalhistas
n = 400
anos = rng.integers(1, 15, n)
valor = rng.gamma(4, 10, n).round(1)
testemunha = rng.binomial(1, .5, n)
acordo = rng.binomial(1, sig(1.7 - 0.06 * valor + 0.15 * anos - 1.0 * testemunha))
trabalhista = pd.DataFrame({"acordo": acordo, "valor_pedido": valor,
                            "anos_empresa": anos, "testemunha": testemunha})

# exercício 2: pedidos de benefício
n = 2000
dias = rng.integers(0, 60, n)
divergencia = rng.binomial(1, .3, n)
procurador = rng.binomial(1, .1, n)
fraude = rng.binomial(1, sig(-5.0 + 0.04 * dias + 1.6 * divergencia + 1.8 * procurador))
beneficios = pd.DataFrame({"fraude": fraude, "dias_cadastro": dias,
                           "divergencia": divergencia, "procurador_frequente": procurador})

# exercício 3: ações contra companhias aéreas
n = 600
problema = rng.choice(["atraso", "cancelamento", "bagagem"], n, p=[.5, .3, .2])
advogado = rng.binomial(1, .5, n)
comprovante = rng.binomial(1, np.where(advogado == 1, .8, .3))
z = (-0.9 + np.select([problema == "cancelamento", problema == "bagagem"], [0.7, 1.2], 0)
     + 0.2 * advogado + 1.5 * comprovante)
aereas = pd.DataFrame({"procedente": rng.binomial(1, sig(z)), "problema": problema,
                       "advogado": advogado, "comprovante": comprovante})

# exercício 4: juizado especial cível
n = 320
juiz = rng.integers(1, 81, n)
efeito_juiz = rng.normal(0, 0.3, 81)
prova = rng.binomial(1, .5, n)
valor = rng.gamma(3, 3, n).round(1)
procedente = rng.binomial(1, sig(-0.8 + 2.0 * prova - 0.04 * valor + efeito_juiz[juiz]))
juizado = pd.DataFrame({"procedente": procedente, "prova_documental": prova,
                        "valor_pedido": valor, "juiz": [f"J{j:02d}" for j in juiz]})
ordem = rng.permutation(n)
treino, teste = juizado.iloc[ordem[:220]], juizado.iloc[ordem[220:]]


def razoes_de_chance(modelo):
    """Coeficiente, razão de chances e valor-p de cada variável."""
    return pd.DataFrame({"coeficiente": modelo.params, "razao_de_chances": np.exp(modelo.params),
                         "valor_p": modelo.pvalues}).round(3)


def medidas_por_corte(y, prob, cortes):
    """Matriz de confusão e medidas para cada corte."""
    linhas = []
    for c in cortes:
        sinal = prob >= c
        vp, fp = int((sinal & (y == 1)).sum()), int((sinal & (y == 0)).sum())
        fn, vn = int((~sinal & (y == 1)).sum()), int((~sinal & (y == 0)).sum())
        linhas.append({"corte": c, "sinalizados": vp + fp, "VP": vp, "FP": fp, "FN": fn, "VN": vn,
                       "acuracia": (vp + vn) / len(y), "precisao": vp / max(vp + fp, 1),
                       "sensibilidade": vp / (vp + fn)})
    return pd.DataFrame(linhas).round(3)


def colunas(dados, com_juiz):
    x = dados[["prova_documental", "valor_pedido"]].copy()
    if com_juiz:
        juizes = sorted(juizado["juiz"].unique())
        x = x.join(pd.get_dummies(dados["juiz"], dtype=float).reindex(columns=juizes, fill_value=0))
    return x


def ajusta_juizado(com_juiz):
    """Regressão logística no treino, com ou sem o juiz como variável."""
    return LogisticRegression(C=1e4, max_iter=20000).fit(colunas(treino, com_juiz), treino["procedente"])


def prob_juizado(modelo, dados, com_juiz):
    return modelo.predict_proba(colunas(dados, com_juiz))[:, 1]


print("Bases criadas: trabalhista, beneficios, aereas, juizado (treino e teste).")

[Volta ao Índice](#indice)

___
<div id="ex1"></div>

## Exercício 1: reclamações trabalhistas, quantas terminam em acordo?

Uma vara do trabalho quer prever quais reclamações vão terminar em **acordo**,
para marcar audiência de conciliação logo no início só nesses processos. Na
base `trabalhista`, cada linha é uma reclamação:

| coluna | o que é |
|---|---|
| `acordo` | 1 se terminou em acordo, 0 se não |
| `valor_pedido` | valor pedido na reclamação, em R\$ mil |
| `anos_empresa` | anos de trabalho na empresa |
| `testemunha` | 1 se o reclamante arrolou testemunha |

A regressão logística já está ajustada. O processo é previsto como acordo
quando a probabilidade é de pelo menos 0,5.

In [ ]:
modelo1 = smf.logit("acordo ~ valor_pedido + anos_empresa + testemunha", data=trabalhista).fit(disp=0)
trabalhista["prob"] = modelo1.predict(trabalhista)
trabalhista["previsto"] = (trabalhista["prob"] >= 0.5).astype(int)

trabalhista["acordo"].mean()          # proporção de acordos na base

In [ ]:
pd.crosstab(trabalhista["acordo"], trabalhista["previsto"], margins=True)   # linhas: real; colunas: previsto

<div id="ex1a"></div>

**(a)** Usando os números da matriz de confusão, calcule a **acurácia**, a
**sensibilidade**, a **especificidade** e a **precisão** do modelo.

In [ ]:
# faça aqui as suas contas (o Python serve de calculadora)

*(escreva aqui a sua resposta)*

<div id="ex1b"></div>

**(b)** Escreva uma frase para a **sensibilidade** e outra para a **precisão**,
no vocabulário do caso (reclamações, acordos, audiências), sem usar as palavras
"sensibilidade" e "precisão".

*(escreva aqui a sua resposta)*

<div id="ex1c"></div>

**(c)** Aqui perto de metade das reclamações (45%) termina em acordo. Um modelo "trivial", que
prevê sempre a mesma resposta para todo mundo, teria que acurácia? A acurácia é
uma boa medida para comparar o modelo com ele neste caso? Por quê?

*(escreva aqui a sua resposta)*

[Volta ao Índice](#indice)

___
<div id="ex2"></div>

## Exercício 2: pedidos de benefício, onde está a fraude?

Uma equipe de auditoria previdenciária recebe **2.000 pedidos de benefício**
por mês e quer separar os suspeitos de **fraude** para análise manual. Na base
`beneficios`:

| coluna | o que é |
|---|---|
| `fraude` | 1 se o pedido era fraudulento, 0 se não |
| `dias_cadastro` | dias entre o cadastro do segurado e o pedido |
| `divergencia` | 1 se houve divergência nos documentos |
| `procurador_frequente` | 1 se o pedido foi feito por procurador com muitos pedidos |

In [ ]:
modelo2 = smf.logit("fraude ~ dias_cadastro + divergencia + procurador_frequente", data=beneficios).fit(disp=0)
beneficios["prob"] = modelo2.predict(beneficios)

beneficios["fraude"].mean()          # proporção de fraudes na base

In [ ]:
medidas_por_corte(beneficios["fraude"], beneficios["prob"], [0.5])

<div id="ex2a"></div>

**(a)** Qual seria a acurácia de um modelo trivial que diz que **nenhum** pedido
é fraude? Compare com a acurácia do modelo no corte 0,5, e com a sensibilidade
dele nesse corte. O modelo com corte 0,5 é útil para a equipe?

In [ ]:
# faça aqui as suas contas (o Python serve de calculadora)

*(escreva aqui a sua resposta)*

<div id="ex2b"></div>

**(b)** A equipe consegue analisar até **350 pedidos por mês**. Rode a tabela abaixo
e escolha o corte que usa essa capacidade. Nesse corte, quantas fraudes a equipe
encontra e quantas deixa passar? De cada 10 pedidos analisados, quantos são
fraude de verdade?

In [ ]:
medidas_por_corte(beneficios["fraude"], beneficios["prob"], [0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50])

*(escreva aqui a sua resposta)*

<div id="ex2c"></div>

**(c)** Qual medida mais importa para a equipe: acurácia, sensibilidade ou
precisão? Justifique com o que cada erro custa (o pedido honesto analisado à toa
e a fraude que passa). Se a equipe dobrasse de tamanho, o que você mudaria?

*(escreva aqui a sua resposta)*

[Volta ao Índice](#indice)

___
<div id="ex3"></div>

## Exercício 3: ações contra companhias aéreas, o que explica a procedência?

Um juizado recebe muitas ações de passageiros contra companhias aéreas, pedindo
indenização. Na base `aereas`:

| coluna | o que é |
|---|---|
| `procedente` | 1 se o pedido foi julgado procedente, 0 se não |
| `problema` | `atraso`, `cancelamento` ou `bagagem` (extravio de bagagem) |
| `advogado` | 1 se o passageiro tinha advogado |
| `comprovante` | 1 se o passageiro juntou comprovantes dos gastos (hotel, alimentação, compras) |

`problema` é **categórica**: o modelo usa `atraso` como categoria de referência,
e cada uma das outras ganha o seu coeficiente, sempre em comparação com o atraso.

In [ ]:
modelo3a = smf.logit("procedente ~ C(problema, Treatment('atraso')) + advogado", data=aereas).fit(disp=0)
razoes_de_chance(modelo3a)

<div id="ex3a"></div>

**(a)** No modelo acima, interprete a razão de chances de **cancelamento** e a de
**bagagem**. Qual tipo de problema tem mais chance de procedência?

*(escreva aqui a sua resposta)*

<div id="ex3b"></div>

**(b)** Agora entra a variável `comprovante`. Compare a razão de chances de
`advogado` nos dois modelos. O que aconteceu, e por quê? A tabela cruzada abaixo
ajuda. Dá para dizer que ter advogado aumenta a chance de ganhar?

In [ ]:
modelo3b = smf.logit("procedente ~ C(problema, Treatment('atraso')) + advogado + comprovante", data=aereas).fit(disp=0)
razoes_de_chance(modelo3b)

In [ ]:
pd.crosstab(aereas["advogado"], aereas["comprovante"], normalize="index")   # proporção com comprovante, por advogado

*(escreva aqui a sua resposta)*

<div id="ex3c"></div>

**(c)** Com os **coeficientes** do segundo modelo, calcule a probabilidade
prevista de procedência para um passageiro com **voo cancelado**, **sem
advogado** e **com comprovante**. Lembre: some o intercepto com os coeficientes
que se aplicam ao caso, e use $p = \dfrac{1}{1 + e^{-\text{soma}}}$
(`np.exp` faz a exponencial). Confira com `modelo3b.predict()` se quiser.

In [ ]:
# faça aqui as suas contas (o Python serve de calculadora)

*(escreva aqui a sua resposta)*

[Volta ao Índice](#indice)

___
<div id="ex4"></div>

## Exercício 4: juizado especial, e se o modelo souber quem é o juiz?

Um escritório quer prever se ações de indenização no juizado especial cível
serão **procedentes**. Na base `juizado` há 320 processos, julgados por **80
juízes** diferentes:

| coluna | o que é |
|---|---|
| `procedente` | 1 se o pedido foi julgado procedente |
| `prova_documental` | 1 se a inicial veio com prova documental |
| `valor_pedido` | valor pedido, em R\$ mil |
| `juiz` | quem julgou (`J01` a `J80`) |

Os processos já foram separados em **treino** (220, para ajustar o modelo) e
**teste** (100, que o modelo não vê, para conferir o acerto em processos
novos). Comparamos dois modelos: o **A** usa só a prova e o valor; o **B** usa
também o juiz, uma variável categórica com 80 categorias.

In [ ]:
linhas = []
for nome, com_juiz in [("A: prova e valor", False), ("B: prova, valor e juiz", True)]:
    modelo = ajusta_juizado(com_juiz)
    acerto_treino = ((prob_juizado(modelo, treino, com_juiz) >= 0.5) == treino["procedente"]).mean()
    acerto_teste = ((prob_juizado(modelo, teste, com_juiz) >= 0.5) == teste["procedente"]).mean()
    linhas.append({"modelo": nome, "acuracia_treino": acerto_treino, "acuracia_teste": acerto_teste})
pd.DataFrame(linhas)

<div id="ex4a"></div>

**(a)** Qual modelo acerta mais **no treino**? E **no teste**? Qual você usaria
para prever processos novos, e por quê?

*(escreva aqui a sua resposta)*

<div id="ex4b"></div>

**(b)** A tabela abaixo mostra, para alguns juízes, quantos processos cada um
tem no treino, quantos foram procedentes, e a probabilidade que o modelo B dá a
um processo **novo** desse juiz, com prova documental e valor de R\$ 10 mil.
Você confiaria nessas probabilidades? Por quê?

In [ ]:
modeloB = ajusta_juizado(com_juiz=True)
por_juiz = treino.groupby("juiz")["procedente"].agg(processos="count", procedentes="sum")
novo = pd.DataFrame({"prova_documental": 1, "valor_pedido": 10.0, "juiz": por_juiz.index})
por_juiz["prob_modelo_B"] = prob_juizado(modeloB, novo, com_juiz=True)
por_juiz.sort_values("prob_modelo_B").iloc[[0, 1, 2, -3, -2, -1]].round(3)

*(escreva aqui a sua resposta)*

<div id="ex4c"></div>

**(c)** Explique, com as suas palavras, por que o modelo B foi pior no teste,
mesmo tendo mais informação. O que você faria para usar a informação do juiz
sem cair nesse problema? (É o assunto das próximas aulas: dê o seu palpite.)

*(escreva aqui a sua resposta)*

[Volta ao Índice](#indice)